In [ ]:
!pip install ultralytics -q
print("Ultralytics is installed")

In [ ]:
import torch
assert torch.cuda.is_available(),
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
from pathlib import Path

DATASET_DIR = "/kaggle/input/datasets/natair/chicken-dataset/chicken_dataset"

WORK_DIR    = "/kaggle/working"
RUN_NAME    = "chicken_detector"
IMG_SIZE    = 1280
TOTAL_EPOCHS = 150
BATCH       = 8
MODEL_BASE  = "yolov8m.pt"

In [ ]:
# Creating a dataset config for YOLO

data_yaml = f"""
path: {DATASET_DIR}
train: images/train
val: images/val

names:
  0: chicken
"""

yaml_path = f"{WORK_DIR}/data.yaml"
with open(yaml_path, "w") as f:
    f.write(data_yaml)

print(f"data.yaml created: {yaml_path}")
print(data_yaml)

train_imgs = list(Path(DATASET_DIR, "images/train").glob("*.jpg"))
val_imgs   = list(Path(DATASET_DIR, "images/val").glob("*.jpg"))
train_lbls = list(Path(DATASET_DIR, "labels/train").glob("*.txt"))
val_lbls   = list(Path(DATASET_DIR, "labels/val").glob("*.txt"))

print(f"Train: {len(train_imgs)} images / {len(train_lbls)} annotaions")
print(f"Val:   {len(val_imgs)} images / {len(val_lbls)} annotaions")

assert len(train_imgs) > 0, "No train images found."
assert len(train_imgs) == len(train_lbls), "The number of images and annotations does not match!"

In [ ]:
from ultralytics import YOLO
import os

RESUME_CHECKPOINT = None
# "/kaggle/input/chicken-checkpoint/last.pt"

if RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).exists():
    print(f"Continuing training from the checkpoint: {RESUME_CHECKPOINT}")
    model = YOLO(RESUME_CHECKPOINT)
    resume_flag = True
else:
    print(f"Start training from scratch: {MODEL_BASE}")
    model = YOLO(MODEL_BASE)
    resume_flag = False

results = model.train(
    data=yaml_path,
    epochs=TOTAL_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    name=RUN_NAME,
    project=f"{WORK_DIR}/runs",
    resume=resume_flag,

    # Augmentations for small objects
    mosaic=1.0,
    scale=0.9,
    copy_paste=0.3,
    degrees=10.0,
    fliplr=0.5,
    flipud=0.3,
    hsv_h=0.015,
    hsv_v=0.4,

    # Session interruption resilience
    save=True,
    save_period=5,      # checkpoint every 5 epochs
    patience=40,        # early discontinuation if there is no improvement over a prolonged period
    exist_ok=True,      # write to the same 'run' folder again
    verbose=True,
    plots=True,
)

print("\n Training completed (or interrupted due to patience)")
print(f"  best.pt: {WORK_DIR}/runs/{RUN_NAME}/weights/best.pt")
print(f"  last.pt: {WORK_DIR}/runs/{RUN_NAME}/weights/last.pt")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_csv = f"{WORK_DIR}/runs/{RUN_NAME}/results.csv"
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

print("Last 5 epochs:")
print(df[["epoch", "metrics/precision(B)", "metrics/recall(B)",
          "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50")
axes[0].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")
axes[0].set_title("mAP by epochs")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(df["epoch"], df["metrics/precision(B)"], label="Precision")
axes[1].plot(df["epoch"], df["metrics/recall(B)"], label="Recall")
axes[1].set_title("Precision / Recall")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{WORK_DIR}/training_summary.jpg", dpi=120)
plt.show()

best_map50 = df["metrics/mAP50(B)"].max()
print(f"\nBest mAP50: {best_map50:.3f}")

In [ ]:
best_model = YOLO(f"{WORK_DIR}/runs/{RUN_NAME}/weights/best.pt")

# Pick any image from 'val' for a visual check.
test_img = str(val_imgs[0])
res = best_model(test_img, imgsz=IMG_SIZE, conf=0.25)

res[0].save(filename=f"{WORK_DIR}/test_prediction.jpg")
print(f"Result saved: {WORK_DIR}/test_prediction.jpg")
print(f"Objects found: {len(res[0].boxes)}")

import cv2
img = cv2.cvtColor(cv2.imread(f"{WORK_DIR}/test_prediction.jpg"), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 9))
plt.imshow(img)
plt.axis("off")
plt.title(f"Detections: {len(res[0].boxes)}")
plt.show()